# Notebook 05 — Analysis & Frontend
**Student 5 · Analysis + Frontend**

Performs final performance analysis, SHAP feature importance, weight stability,
limitations discussion, generates `frontend.json`, and builds the
`frontend/index.html` FolioWise dashboard.

**Prerequisites:** Run `04_backtest.ipynb` first.

In [ ]:
# ── Cell 1 · Setup & Load All Results ─────────────────────────────────────────
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_DIR     = '../data'
RAW_DIR      = os.path.join(DATA_DIR, 'raw')
PROC_DIR     = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR  = '../results'
FRONTEND_DIR = '../frontend'
os.makedirs(FRONTEND_DIR, exist_ok=True)

# Load all results
strategy_metrics = pd.read_csv(os.path.join(RESULTS_DIR, 'strategy_metrics.csv'))
daily_returns    = pd.read_csv(os.path.join(RESULTS_DIR, 'daily_returns.csv'), parse_dates=['date'])
weights_df       = pd.read_csv(os.path.join(RESULTS_DIR, 'weights.csv'), parse_dates=['date'])
predictions_df   = pd.read_csv(os.path.join(RESULTS_DIR, 'predictions.csv'), parse_dates=['date'])
model_metrics    = pd.read_csv(os.path.join(RESULTS_DIR, 'model_metrics.csv'))
features_df      = pd.read_csv(os.path.join(PROC_DIR, 'features.csv'), parse_dates=['date'])
universe_df      = pd.read_csv(os.path.join(RAW_DIR, 'universe.csv'))
prices_df        = pd.read_csv(os.path.join(RAW_DIR, 'prices.csv'), index_col='date', parse_dates=True)
index_df         = pd.read_csv(os.path.join(RAW_DIR, 'index.csv'), index_col='date', parse_dates=True)

print('All result files loaded.')
print(f'Strategy metrics : {strategy_metrics.shape}')
print(f'Daily returns    : {daily_returns.shape}')
print(f'Weights          : {weights_df.shape}')
print(f'Predictions      : {predictions_df.shape}')

In [ ]:
# ── Cell 2 · Did S4 Beat S1, S2, and the Index? ───────────────────────────────
print('=' * 60)
print('QUESTION: Did S4 (Max Sharpe ML) beat S1 (EW), S2 (MinVar), S0 (NIFTY)?')
print('=' * 60)

# Full period comparison
comp_strats = ['S0', 'S1', 'S2', 'S4']
comp_df     = strategy_metrics[strategy_metrics['strategy'].str.startswith(
    tuple(comp_strats))].copy()
print('\n--- Full Period ---')
print(comp_df[['strategy','cagr_pct','vol_pct','sharpe','max_dd_pct','calmar']].to_string(index=False))

# Sub-period analysis
SUB_PERIODS = {
    'Pre-COVID (2020-2021)': ('2020-01-01', '2021-12-31'),
    'Recovery (2022-2023)': ('2022-01-01', '2023-12-31'),
    'Recent (2024-2026)':   ('2024-01-01', '2026-10-31'),
}

for period_name, (p_start, p_end) in SUB_PERIODS.items():
    print(f'\n--- {period_name} ---')
    period_rows = []
    for sid in comp_strats:
        sub = daily_returns[
            (daily_returns['strategy'] == sid) &
            (daily_returns['date'] >= p_start) &
            (daily_returns['date'] <= p_end)
        ]['return']
        if len(sub) < 20:
            continue
        cagr   = ((1 + sub).prod()) ** (252 / len(sub)) - 1
        vol    = sub.std() * np.sqrt(252)
        sharpe = (sub.mean() * 252) / vol if vol > 0 else np.nan
        period_rows.append({'strategy': sid,
                            'cagr_pct': round(cagr*100,2),
                            'vol_pct': round(vol*100,2),
                            'sharpe': round(sharpe,3)})
    if period_rows:
        print(pd.DataFrame(period_rows).to_string(index=False))

# Verdict
s4 = strategy_metrics[strategy_metrics['strategy'].str.contains('S4')]
s1 = strategy_metrics[strategy_metrics['strategy'].str.contains('S1')]
s0 = strategy_metrics[strategy_metrics['strategy'].str.contains('S0')]

print('\n=== VERDICT ===')
if not s4.empty and not s1.empty:
    if s4.iloc[0]['sharpe'] > s1.iloc[0]['sharpe']:
        print(f'✅ S4 Sharpe ({s4.iloc[0]["sharpe"]:.3f}) > S1 Sharpe ({s1.iloc[0]["sharpe"]:.3f}): S4 BEATS S1')
    else:
        print(f'❌ S4 Sharpe ({s4.iloc[0]["sharpe"]:.3f}) ≤ S1 Sharpe ({s1.iloc[0]["sharpe"]:.3f}): S4 does NOT beat S1')
    if not s0.empty and s4.iloc[0]['cagr_pct'] > s0.iloc[0]['cagr_pct']:
        print(f'✅ S4 CAGR ({s4.iloc[0]["cagr_pct"]:.1f}%) > NIFTY CAGR ({s0.iloc[0]["cagr_pct"]:.1f}%): S4 BEATS INDEX')
    else:
        print('❌ S4 did not outperform NIFTY index on CAGR')

In [ ]:
# ── Cell 3 · SHAP / Feature Importance Analysis ───────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed. Using sklearn feature importance instead.')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [c for c in features_df.columns if c.startswith('f_')]
TARGET_COL   = 'target_21d'
RANDOM_SEED  = 42

clean_df = features_df.dropna(subset=FEATURE_COLS + [TARGET_COL])
X        = clean_df[FEATURE_COLS].values
y        = clean_df[TARGET_COL].values

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

rf = RandomForestRegressor(n_estimators=200, max_depth=3, min_samples_leaf=20,
                           random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_sc, y)

if SHAP_AVAILABLE:
    print('Computing SHAP values on a sample of 2000 rows …')
    sample_idx = np.random.choice(len(X_sc), min(2000, len(X_sc)), replace=False)
    X_sample   = X_sc[sample_idx]
    explainer  = shap.TreeExplainer(rf)
    shap_vals  = explainer.shap_values(X_sample)

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_sample, feature_names=FEATURE_COLS,
                      plot_type='bar', show=False)
    plt.title('SHAP Feature Importance (RF, bar)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'shap_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP bar plot saved → results/shap_importance.png')

    fig2, _ = plt.subplots(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_sample, feature_names=FEATURE_COLS, show=False)
    plt.title('SHAP Beeswarm', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'shap_beeswarm.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP beeswarm saved → results/shap_beeswarm.png')
else:
    imp_df = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    imp_df.plot.barh(ax=ax, color='steelblue')
    ax.set_title('RF Feature Importance (sklearn)', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'feature_importances_05.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 4 · Weight Stability Analysis ───────────────────────────────────────
print('=== Weight Stability Analysis ===')

for sid in ['S1', 'S2', 'S4', 'S6']:
    w_sub = weights_df[weights_df['strategy'] == sid].copy()
    if w_sub.empty:
        continue
    # Pivot to wide
    w_wide = w_sub.pivot_table(index='date', columns='ticker', values='weight', fill_value=0)
    # Compute month-to-month turnover
    turnover = w_wide.diff().abs().sum(axis=1).mean()
    hhi      = (w_wide ** 2).sum(axis=1).mean()  # Herfindahl concentration
    # Effective N
    eff_n    = (1 / hhi) if hhi > 0 else np.nan
    print(f'  {sid}: Avg Turnover={turnover:.3f}  HHI={hhi:.4f}  Effective N={eff_n:.1f}')

# Plot weight concentration over time for S4
s4_w = weights_df[weights_df['strategy'] == 'S4'].copy()
if not s4_w.empty:
    s4_wide  = s4_w.pivot_table(index='date', columns='ticker', values='weight', fill_value=0)
    hhi_ts   = (s4_wide ** 2).sum(axis=1)
    eff_n_ts = 1 / hhi_ts

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    hhi_ts.plot(ax=axes[0], color='darkred')
    axes[0].set_title('S4 Portfolio Concentration (HHI) Over Time')
    axes[0].set_ylabel('HHI')

    eff_n_ts.plot(ax=axes[1], color='steelblue')
    axes[1].set_title('S4 Effective Number of Holdings Over Time')
    axes[1].set_ylabel('Effective N')

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'weight_stability.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → results/weight_stability.png')

In [ ]:
# ── Cell 5 · Limitations Analysis ─────────────────────────────────────────────
limitations = """
## Known Limitations of This Study

### 1. Survivorship Bias
The universe consists of *current* NIFTY 50 constituents, all of which survived
from 2015 to 2026. Stocks that were delisted or removed from the index during
this period are excluded. This inflates backtested returns relative to what a
real investor would have achieved.

### 2. Transaction Costs & Slippage
We model transaction costs at 0.2% per side of turnover. Real-world costs for
institutional investors include market impact, borrow costs for any shorts,
STT, and exchange fees. All of these are ignored or understated here.

### 3. Look-Ahead Bias
Although a formal leakage test was run (Notebook 02), subtle look-ahead can
remain in: (a) universe selection (we knew which stocks survived), (b) the
NIFTY benchmark dates used for alignment.

### 4. Short Backtest Period
11 years (2015–2026) covers only two full market cycles. Many ML effects
require longer out-of-sample validation to be statistically meaningful.

### 5. Feature Stationarity
Returns, RSI, and momentum features are not guaranteed to be stationary over
long horizons. The model may degrade in market regimes not represented in
training data.

### 6. Cross-Sectional vs. Absolute Signals
All models are trained as cross-sectional rankers. The portfolio goes long
even when every stock has negative predicted returns — there is no market
timing or cash-out mechanism.

### 7. Hyperparameter Search on Out-of-Sample Data
The Ridge alpha grid, RF depth, and XGB settings were chosen based on
published literature, but final evaluation was done on the same data used
for model selection decisions. A truly held-out final test set was not retained.
"""
print(limitations)

In [ ]:
# ── Cell 6 · Build frontend.json ──────────────────────────────────────────────
def safe_float(val):
    """Convert to Python float, replacing NaN/inf with None."""
    try:
        f = float(val)
        return None if (np.isnan(f) or np.isinf(f)) else round(f, 4)
    except:
        return None

# Build equity curve data for each strategy
equity_curves = {}
for sid in ['S0','S1','S2','S3','S4','S5','S6']:
    sub = daily_returns[daily_returns['strategy'] == sid].sort_values('date')
    if sub.empty:
        continue
    eq  = (1 + sub['return']).cumprod()
    equity_curves[sid] = {
        'dates':  [d.strftime('%Y-%m-%d') for d in sub['date']],
        'values': [safe_float(v) for v in eq.values]
    }

# Strategy metrics for display
strategy_table = []
for _, row in strategy_metrics.iterrows():
    strategy_table.append({
        'strategy':     str(row.get('strategy','')),
        'cagr_pct':     safe_float(row.get('cagr_pct')),
        'vol_pct':      safe_float(row.get('vol_pct')),
        'sharpe':       safe_float(row.get('sharpe')),
        'max_dd_pct':   safe_float(row.get('max_dd_pct')),
        'calmar':       safe_float(row.get('calmar')),
        'avg_turnover': safe_float(row.get('avg_turnover')),
    })

# Model metrics
model_table = []
for _, row in model_metrics.iterrows():
    model_table.append({
        'model':            str(row.get('model','')),
        'ic_mean':          safe_float(row.get('ic_mean')),
        'ic_tstat':         safe_float(row.get('ic_tstat')),
        'rmse':             safe_float(row.get('rmse')),
        'dir_acc':          safe_float(row.get('dir_acc')),
        'top_minus_bottom': safe_float(row.get('top_minus_bottom')),
    })

# Top holdings from latest S4 weights
top_holdings = []
s4_latest = weights_df[weights_df['strategy'] == 'S4'].sort_values('date')
if not s4_latest.empty:
    latest_date = s4_latest['date'].max()
    latest_w    = s4_latest[s4_latest['date'] == latest_date].sort_values('weight', ascending=False)
    for _, row in latest_w.head(10).iterrows():
        tkr  = row['ticker']
        info = universe_df[universe_df['ticker'] == tkr]
        name = info.iloc[0]['name'] if not info.empty else tkr
        sec  = info.iloc[0]['sector'] if not info.empty else ''
        top_holdings.append({
            'ticker': tkr, 'name': name, 'sector': sec,
            'weight_pct': safe_float(row['weight'] * 100)
        })

frontend_json = {
    'generated_at':   pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'project':        'FolioWise — ML Portfolio Optimizer',
    'universe_size':  len(universe_df),
    'tickers':        universe_df['ticker'].tolist(),
    'strategy_labels': {
        'S0': 'NIFTY Benchmark', 'S1': 'Equal Weight',
        'S2': 'Min Variance',    'S3': 'Max Sharpe (Historical)',
        'S4': 'Max Sharpe (ML)', 'S5': 'Top-5 ML Equal Weight',
        'S6': 'HRP',
    },
    'equity_curves':  equity_curves,
    'strategy_table': strategy_table,
    'model_table':    model_table,
    'top_holdings':   top_holdings,
    'limitations':    limitations.strip(),
}

json_path = os.path.join(FRONTEND_DIR, 'frontend.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(frontend_json, f, indent=2)
print(f'Saved → {json_path}')

In [ ]:
# ── Cell 7 · Build Frontend (index.html) ──────────────────────────────────────
build_script = os.path.abspath('../src/build_frontend.py')

if os.path.exists(build_script):
    import subprocess
    result = subprocess.run(
        ['python', build_script],
        capture_output=True, text=True,
        cwd=os.path.abspath('..')
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
    html_path = os.path.join(FRONTEND_DIR, 'index.html')
    if os.path.exists(html_path):
        print(f'✅  Frontend built → {html_path}')
    else:
        print('⚠️  build_frontend.py ran but index.html not found. Generating inline …')
else:
    print('build_frontend.py not found — generating a simple HTML dashboard inline …')

# Always generate a self-contained fallback / supplement HTML
html_template = '''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>FolioWise — ML Portfolio Optimizer</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Segoe UI', sans-serif; background: #0f172a; color: #e2e8f0; }
    header { background: linear-gradient(135deg,#1e3a5f,#0d9488); padding: 24px 40px; }
    header h1 { font-size: 2rem; color: #fff; }
    header p { color: #94a3b8; margin-top: 4px; }
    .container { max-width: 1200px; margin: 0 auto; padding: 32px 24px; }
    .card { background: #1e293b; border-radius: 12px; padding: 24px; margin-bottom: 24px; }
    .card h2 { font-size: 1.2rem; color: #0d9488; margin-bottom: 16px; }
    table { width: 100%; border-collapse: collapse; font-size: 0.85rem; }
    th { background: #0f172a; padding: 10px; text-align: right; color: #94a3b8; border-bottom: 1px solid #334155; }
    th:first-child { text-align: left; }
    td { padding: 9px 10px; border-bottom: 1px solid #1e293b; text-align: right; }
    td:first-child { text-align: left; }
    tr:hover { background: #334155; }
    .best { color: #22d3ee; font-weight: bold; }
    canvas { max-height: 380px; }
    .grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
    @media(max-width:768px){ .grid2 { grid-template-columns: 1fr; } }
    footer { text-align: center; padding: 20px; color: #475569; font-size: 0.75rem; }
    .badge { display:inline-block; padding:2px 8px; border-radius:99px; font-size:0.7rem;
             background:#0d9488; color:#fff; margin-left:8px; }
    .lim { background: #0f172a; padding: 16px; border-radius: 8px;
           border-left: 4px solid #f59e0b; white-space: pre-wrap;
           font-size: 0.82rem; color: #cbd5e1; line-height: 1.6; }
  </style>
</head>
<body>
<header>
  <h1>📊 FolioWise <span class="badge">ML Portfolio Optimizer</span></h1>
  <p>NIFTY 50 · Walk-Forward ML · 2020–2026 Out-of-Sample</p>
</header>
<div class="container">

  <div class="card">
    <h2>⚡ Equity Curves (Cumulative Returns)</h2>
    <canvas id="equityChart"></canvas>
  </div>

  <div class="card">
    <h2>🏆 Strategy Performance Summary</h2>
    <table id="stratTable">
      <thead><tr>
        <th>Strategy</th><th>CAGR %</th><th>Vol %</th>
        <th>Sharpe</th><th>Max DD %</th><th>Calmar</th><th>Avg TO %</th>
      </tr></thead>
      <tbody></tbody>
    </table>
  </div>

  <div class="grid2">
    <div class="card">
      <h2>🤖 ML Model Metrics</h2>
      <table id="modelTable">
        <thead><tr>
          <th>Model</th><th>IC Mean</th><th>IC t-stat</th>
          <th>RMSE</th><th>Dir Acc</th><th>TMB</th>
        </tr></thead>
        <tbody></tbody>
      </table>
    </div>
    <div class="card">
      <h2>🎯 S4 Top Holdings (Latest)</h2>
      <table id="holdTable">
        <thead><tr><th>Ticker</th><th>Name</th><th>Sector</th><th>Weight %</th></tr></thead>
        <tbody></tbody>
      </table>
    </div>
  </div>

  <div class="card">
    <h2>⚠️ Limitations</h2>
    <div class="lim" id="limitations"></div>
  </div>

</div>
<footer>Generated by FolioWise · Minor Project · {generated_at}</footer>

<script>
const DATA = {json_data};

// ── Equity Chart ──────────────────────────────────────────────────────────
const COLORS = ['#94a3b8','#f59e0b','#22d3ee','#a78bfa','#f43f5e','#4ade80','#fb923c'];
const sids   = Object.keys(DATA.equity_curves);
const labels = DATA.equity_curves[sids[0]].dates;
// Sample to ~100 points
const step  = Math.max(1, Math.floor(labels.length / 100));
const sampled_labels = labels.filter((_,i) => i % step === 0);

const datasets = sids.map((sid, i) => ({
  label: DATA.strategy_labels[sid] || sid,
  data : DATA.equity_curves[sid].values.filter((_,j) => j % step === 0),
  borderColor: COLORS[i % COLORS.length],
  backgroundColor: 'transparent',
  borderWidth: sid === 'S4' ? 3 : 1.5,
  pointRadius: 0,
  tension: 0.3,
}));

new Chart(document.getElementById('equityChart'), {
  type: 'line',
  data: { labels: sampled_labels, datasets },
  options: {
    responsive: true,
    plugins: { legend: { labels: { color: '#cbd5e1', boxWidth: 20 } } },
    scales: {
      x: { ticks: { color: '#64748b', maxTicksLimit: 8 }, grid: { color: '#1e293b' } },
      y: { ticks: { color: '#64748b' }, grid: { color: '#1e293b' } }
    }
  }
});

// ── Strategy Table ────────────────────────────────────────────────────────
const stBody = document.querySelector('#stratTable tbody');
const bestSharpe = Math.max(...DATA.strategy_table.map(r => r.sharpe || -99));
DATA.strategy_table.forEach(r => {
  const best = r.sharpe === bestSharpe;
  stBody.innerHTML += `<tr>
    <td class="${best?'best':''}">${r.strategy}</td>
    <td class="${best?'best':''}">${r.cagr_pct ?? '—'}</td>
    <td>${r.vol_pct ?? '—'}</td>
    <td class="${best?'best':''}">${r.sharpe ?? '—'}</td>
    <td>${r.max_dd_pct ?? '—'}</td>
    <td>${r.calmar ?? '—'}</td>
    <td>${r.avg_turnover ?? '—'}</td>
  </tr>`;
});

// ── Model Table ────────────────────────────────────────────────────────────
const mBody = document.querySelector('#modelTable tbody');
DATA.model_table.forEach(r => {
  mBody.innerHTML += `<tr>
    <td>${r.model}</td><td>${r.ic_mean ?? '—'}</td>
    <td>${r.ic_tstat ?? '—'}</td><td>${r.rmse ?? '—'}</td>
    <td>${r.dir_acc ?? '—'}</td><td>${r.top_minus_bottom ?? '—'}</td>
  </tr>`;
});

// ── Holdings Table ─────────────────────────────────────────────────────────
const hBody = document.querySelector('#holdTable tbody');
DATA.top_holdings.forEach(r => {
  hBody.innerHTML += `<tr>
    <td>${r.ticker}</td><td>${r.name}</td>
    <td>${r.sector}</td><td>${r.weight_pct ?? '—'}</td>
  </tr>`;
});

// ── Limitations ────────────────────────────────────────────────────────────
document.getElementById('limitations').textContent = DATA.limitations;
</script>
</body>
</html>
'''

html_out = html_template.replace(
    '{json_data}', json.dumps(frontend_json, default=str)
).replace('{generated_at}', frontend_json['generated_at'])

html_path = os.path.join(FRONTEND_DIR, 'index.html')
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_out)
print(f'✅  Frontend saved → {html_path}')
print(f'   Open in browser: file://{os.path.abspath(html_path)}')

In [ ]:
# ── Cell 8 · Final Summary Table ──────────────────────────────────────────────
print('=' * 65)
print('               FINAL PROJECT SUMMARY')
print('=' * 65)
print(f'Universe    : {len(universe_df)} stocks (NIFTY 50 constituents)')
print(f'Period      : 2015-01-01 → 2026-11-01  ({len(prices_df)} trading days)')
print(f'Features    : {len([c for c in features_df.columns if c.startswith("f_")])} feature columns')
print(f'Models      : Naive Baseline | Ridge | Random Forest | XGBoost')
print(f'Strategies  : S0 (Benchmark) | S1 (EW) | S2 (MinVar) | S3 (MaxSharpe-Hist)')
print(f'              S4 (MaxSharpe-ML) | S5 (Top5-EW) | S6 (HRP)')
print()
print('--- Strategy Performance ---')
print(strategy_metrics[['strategy','cagr_pct','sharpe','max_dd_pct']].to_string(index=False))
print()
print('--- ML Model IC ---')
print(model_metrics[['model','ic_mean','ic_tstat','dir_acc']].to_string(index=False))
print('=' * 65)
print('\n✅  Notebook 05 complete — all outputs saved.')
print(f'   Open frontend → file://{os.path.abspath(html_path)}')